# ♟️ Chess Numba — Classical HCE Texel / SPSA 分區調參工作台

在完成 `evaluation.py` 的 **8 項錯誤修復與順序修正**（`BUG-1..8`、`ORD-1..3`、`KP-DISCONT`）後，本 Notebook 提供**單一 Kernel、免重載資料、免重編譯 Numba** 的互動式區塊調參流程。

### 🛡️ 三層防過擬合與記憶體安全設計
1. **`dataset_mix03_20m_s2.npz`（20,000,000 局面）無洩漏三切分**：
   - **`17,000,000` Train Pool**：僅供 SPSA mini-batch 梯度估計。
   - **`1,000,000` Dev Split (`rng=0`)**：僅供 SPSA 迭代過程報告與 early stopping 挑選該輪最佳點。
   - **`2,000,000` Unseen Holdout (`rng=42`)**：**完全不進訓練、也不進 early stop**，作為第一道靜態驗收閘門。
2. **`val_pgn.npz`（~1,593,187 唯一靜態局面，來自本引擎 42 份 ≥150k nodes 實戰 PGN、18,608 局）**：
   - **只作獨立驗證集**，絕不參與訓練。
   - 完整保留真實 **`castling_rights`**（104,249 局面含易位權）、合法 **`ep_square`**、**`halfmove_clock` (rule50)**，專門抓出 Bullet `.bin` 缺乏易位權時在 `SHELTER` / `STORM` / `TRAPPED_ROOK` / 王 `PST` 上的盲點。
   - 同時報告 `K=1.0` 與「僅用 baseline 擬合的校準 $K^*$」下的 paired MSE、SE 與 z-score。
3. **單行程常駐記憶體 (`TexelTuningSession`)**：
   - 20M 主資料集（~3.44 GB）+ PGN 驗證集（~0.28 GB）只在 Kernel 載入一次（共 ~3.72 GB），**絕不在背景再開子行程載入第二份**，避免 16 GB 機器發生 Swap / OOM。
   - 接受某區塊 (`session.accept(...)`) 時，自動以 AST 精確行段寫入 `chess_engine/classical/constants.py`（完整保留所有註解與空行）、更新 `constants_snapshot.py`、清除過期 classical Numba 快取，並在記憶體即時更新下一輪 `theta0`（**0 秒重編譯**）。

### 🚦 每個區塊的操作方式與驗收標準
- 每個區塊都拆成兩個相鄰 Cell：**`[Run]` 執行調參與雙重驗證** → 看完報告滿意後再跑 **`[Accept]` 套用並備份**（不滿意直接跳過或調小 `c` / `alpha` 重跑即可，不會汙染 `constants.py`）。
- **Gate Verdict 判準**：
  - `PASS`：`2M mix03 holdout` 總 MSE 顯著下降（`delta < 0`）、`broad-EG (phase<=64)` 未退步，且獨立 `val_pgn` 在 $K^*$ 未出現顯著退步（涉及易位/兵盾區塊時 `castling!=0` 亦穩定）。
  - `CAUTION`：主驗證集改善，但某子桶或 `val_pgn` 微幅正值（$|z| < 2$）；建議檢視變動幅度或搭配對局測試決定。
  - `REJECT`：Unseen holdout 退步、`broad-EG` 顯著惡化（$z > +1.8$），或 `val_pgn` 雙尺度顯著惡化（$z > +2.0$）→ **請勿執行 Accept**。

In [ ]:
# 0. 環境檢查與載入 TexelTuningSession（全流程只需執行一次，冷啟動含載入與快取分數約 25~45 秒）
import os
import sys
from pathlib import Path

# 自動定位專案根目錄（相容本機與 Colab）
if Path("chess_engine").is_dir() and Path("tuner").is_dir():
    REPO_ROOT = Path.cwd()
elif (Path.cwd().parent / "chess_engine").is_dir() and Path.cwd().name == "tuner":
    # Notebook 位於 tuner/ 時：切回專案根目錄，確保相對路徑與 runs/ 輸出一致
    REPO_ROOT = Path.cwd().parent
    os.chdir(REPO_ROOT)
elif Path("/content/chess_numba").is_dir():
    REPO_ROOT = Path("/content/chess_numba")
    os.chdir(REPO_ROOT)
else:
    raise RuntimeError(f"請在專案根目錄啟動 Notebook，目前目錄：{Path.cwd()}")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from tuner.notebook_session import TexelTuningSession

# 2026-11-01：改用含真實易位權 / EP / rule50 的 resolved EPD 純 WDL 資料集
# （E12.33 + E12.41 + E12.52 + lichess-big3，去重後抽樣 20M；由 tuner.data_pipeline.convert_epd_books 產生）
TRAIN_NPZ = REPO_ROOT / "tuner" / "data" / "hce" / "dataset_epd_wdl_20m.npz"
PGN_VAL_NPZ = REPO_ROOT / "tuner" / "data" / "hce" / "val_pgn.npz"

# 若尚未建立 val_pgn.npz，可取消下一行註解自動建立（約 40 秒）：
# if not PGN_VAL_NPZ.is_file():
#     from tuner.data_pipeline.build_pgn_validation import build_pgn_validation
#     build_pgn_validation(output_path=PGN_VAL_NPZ, min_median_nodes=150_000, workers=10)

# "auto": 若本機可用實體 RAM < 4.8 GB 自動取 10M train + 完整 1M dev + 2M holdout 防 Swap；設 None 強制載滿 17M train
MAX_TRAIN_ROWS = "auto"

session = TexelTuningSession(
    train_npz=TRAIN_NPZ,
    pgn_val_npz=PGN_VAL_NPZ,
    eval_sample=1_000_000,     # SPSA in-run dev / early-stop
    holdout_sample=2_000_000,  # 完全未見的訓練來源 holdout
    max_train_rows=MAX_TRAIN_ROWS,
    chunk_size=500_000,
)

In [2]:
# 0b. 檢視目前 Baseline 在兩個獨立驗證集上的健康度與最佳 K*
baseline_stats = session.baseline_report()

=== Current Step Baseline Health Report ===
  [1] dataset_epd_wdl_20m.npz unseen holdout (n=2,000,000):
      MSE @ K=1.000 : 0.0951010
      MSE @ K*=1.247: 0.0943020
  [2] val_pgn.npz independent PGN val (n=1,593,187):
      Static Eval MSE @ K=1.000 : 0.0585685
      Static Eval MSE @ K*=1.260: 0.0576835
      Castling!=0 MSE @ K*      : 0.1098110 (n=104,249)
      300k Search Ceiling @ K*=1.477: 0.0445078 (gap=+0.0131757)


In [3]:
# 0c. 熱重載調參器核心（若修改過 tuner/*.py，無須重啟 Kernel 或重載 20M 資料集即可生效）
import importlib
import tuner.tuner as _tuner_mod
import tuner.notebook_session as _ns_mod

importlib.reload(_tuner_mod)
importlib.reload(_ns_mod)
if "session" in globals():
    session.__class__ = _ns_mod.TexelTuningSession
print("✅ 已載入最新 Scale-Aware SPSA (c_floor=1.25, auto_scale=True, EMA + int快取) 與 Gate 判準！")

✅ 已載入最新 Scale-Aware SPSA (c_floor=1.25, auto_scale=True, EMA + int快取) 與 Gate 判準！


---
## Block 1：通路兵系統（Passed Pawns — 對應 `ORD-1..3` 與 `KP-DISCONT` 修復後最優先）
`ORD-1..3` 修正了通路兵路徑安全與攻擊遮罩計算順序，`KP-DISCONT` 補齊多兵純王兵殘局縮放。過去在舊順序下調出的通路兵獎勵與安全權重最需要先重新對齊：
- **Block 1a**：連續 centipawn 表（`PASSED_PAWN_BONUS`、`PASSED_FILE_BONUS`、`PASSED_CONNECTED_BONUS`、`PASSED_PHALANX_BONUS`）。
- **Block 1b**：離散動態公式係數（`PASSED_DYNAMICS_MULT`、`PASSED_DYNAMICS_OFFSET`、`KING_PROX_FRIENDLY_MULT`）先用窮舉掃描，再雙重驗證。
- **Block 1c**：小範圍路徑安全與王距離權重（`PASSED_PATH_*`、`KING_PROX_*`），使用整數安全保底步長 `c=2.0 (c_floor=1.25), alpha=35000`。

In [4]:
# [Run] Block 1a — 通路兵主獎勵表 (CP Tables)
res_1a = session.run_block(
    "block1a_passed_tables",
    tune=[
        "PASSED_PAWN_BONUS",
        "PASSED_FILE_BONUS",
        "PASSED_CONNECTED_BONUS",
        "PASSED_PHALANX_BONUS",
    ],
    iters=3500,
    c=2.5,
    alpha=90000.0,
    batch_size=65536,
    report_every=50,
    patience=25,
    seed=101,
)

=== Running Block [block1a_passed_tables] -> tuner\runs\block1a_passed_tables.py ===
Using preloaded data from dataset_epd_wdl_20m.npz...
Constraints configured: 1203 constraints active.
Loaded 19627749 positions.
Label stats: min=0.0000, max=1.0000, mean=0.4998, std=0.3841
  Continuous/mixed labels detected (102 unique values).
Freezing all parameters except: ['PASSED_PAWN_BONUS', 'PASSED_FILE_BONUS', 'PASSED_CONNECTED_BONUS', 'PASSED_PHALANX_BONUS']
Active parameters: 26 / 1272
Coordinate auto-scale active: min=1.00, median=1.00, max=4.00
Dev val: 1,000,000  |  train pool: 16,627,749  (dev/holdout never enter mini-batches)
Bucket metrics: 250,000 fixed validation positions (membership from initial theta0)
Initial val MSE (K=1.000): 0.0953358
  Fixed-theta0 validation buckets (MSE, n):
    phase=0               0.0578379  n=74,354
    phase=1..31           0.0784561  n=38,264
    phase=32..63          0.0976802  n=33,253
    phase=64..95          0.1211362  n=53,464
    phase=96..128 

In [5]:
# [Accept] Block 1a — 確認上方為 PASS 後再執行本 Cell 套用至 constants.py
session.accept("block1a_passed_tables")

  PASSED_PAWN_BONUS: [0, 0, 1, 3, 0, 0] -> [0, 0, 0, 8, 0, 10]
  PASSED_FILE_BONUS: [16, 4] -> [14, 1]
  PASSED_CONNECTED_BONUS: [0, 2, 3, 7, 17, 23] -> [0, 4, 6, 9, 17, 24]
  PASSED_PHALANX_BONUS: [0, 1, 3, 12, 26, 42] -> [0, 1, 5, 12, 25, 43]
Summary: 4 numeric change(s), 105 identical (skipped), 109 names in tuned dump.
Updated 4 constant(s) in chess_engine\classical\constants.py
Next: python -m tuner.sync_constants_snapshot, light tests, then optional tournament vs classical_old.
Wrote tuner\constants_snapshot.py from chess_engine\classical\constants.py; removed 159 stale classical Numba cache file(s)
[block1a_passed_tables] Accepted 4 parameter group(s). Backup saved to tuner\runs\backups\20261101_203911_before_block1a_passed_tables_constants.py.


['PASSED_PAWN_BONUS',
 'PASSED_FILE_BONUS',
 'PASSED_CONNECTED_BONUS',
 'PASSED_PHALANX_BONUS']

In [6]:
# [Run] Block 1b — 通路兵動態倍率離散窮舉 (w = MULT*rank + OFFSET & 友王距離乘數)
res_1b = session.scan_grid(
    "block1b_passed_dynamics_grid",
    grids=[
        "PASSED_DYNAMICS_MULT=3:5",
        "PASSED_DYNAMICS_OFFSET=-12:-10",
        "KING_PROX_FRIENDLY_MULT=1:2",
    ],
    split="dev",
    eval_sample=1_000_000,
    top=10,
)

=== Discrete Grid Scan [block1b_passed_dynamics_grid] -> tuner\runs\block1b_passed_dynamics_grid.py ===
Scan sample (dev): 1,000,000 (K=1.000); excluded fixed validation=0
  evaluated   1/18: (3, -12, 1)
  evaluated   2/18: (3, -12, 2)
  evaluated   3/18: (3, -11, 1)
  evaluated   4/18: (3, -11, 2)
  evaluated   5/18: (3, -10, 1)
  evaluated   6/18: (3, -10, 2)
  evaluated   7/18: (4, -12, 1)
  evaluated   8/18: (4, -12, 2)
  evaluated   9/18: (4, -11, 1)
  evaluated  10/18: (4, -11, 2)
  evaluated  11/18: (4, -10, 1)
  evaluated  12/18: (4, -10, 2)
  evaluated  13/18: (5, -12, 1)
  evaluated  14/18: (5, -12, 2)
  evaluated  15/18: (5, -11, 1)
  evaluated  16/18: (5, -11, 2)
  evaluated  17/18: (5, -10, 1)
  evaluated  18/18: (5, -10, 2)
Candidates sorted by overall delta (negative is better):
  PASSED_DYNAMICS_MULT PASSED_DYNAMICS_OFFSET KING_PROX_FRIENDLY_MULT          all     broad-EG      phase=0    queenless     minor-EG      rook-EG
             4          -10            1 -2.289

In [7]:
# [Accept] Block 1b — 若離散掃描在 holdout 與 val_pgn 皆穩定改善再套用
session.accept("block1b_passed_dynamics_grid")

  PASSED_DYNAMICS_OFFSET: [-11] -> [-10]
Summary: 1 numeric change(s), 108 identical (skipped), 109 names in tuned dump.
Updated 1 constant(s) in chess_engine\classical\constants.py
Next: python -m tuner.sync_constants_snapshot, light tests, then optional tournament vs classical_old.
Wrote tuner\constants_snapshot.py from chess_engine\classical\constants.py; removed 0 stale classical Numba cache file(s)
[block1b_passed_dynamics_grid] Accepted 1 parameter group(s). Backup saved to tuner\runs\backups\20261101_203943_before_block1b_passed_dynamics_grid_constants.py.


['PASSED_DYNAMICS_OFFSET']

In [8]:
# [Run] Block 1c — 通路兵路徑安全與王距離微調 (Small-Band Knobs)
res_1c = session.run_block(
    "block1c_passed_path_prox",
    tune=[
        "PASSED_PATH_SAFE_NONE_ATTACK",
        "PASSED_PATH_SAFE_EDGE_ATTACK",
        "PASSED_PATH_SAFE_BLOCK_ONLY",
        "PASSED_PATH_SUPPORT_BONUS",
        "KING_PROXIMITY_MAX_DIST",
        "KING_PROX_ENEMY_MULT",
        "KING_PROX_ENEMY_DIV",
        "KING_PROX_MIN_BONUS",
        "KING_PROX_MAX_BONUS",
    ],
    iters=2500,
    c=2.0,
    alpha=35000.0,
    batch_size=65536,
    report_every=50,
    patience=20,
    seed=103,
)

=== Running Block [block1c_passed_path_prox] -> tuner\runs\block1c_passed_path_prox.py ===
Using preloaded data from dataset_epd_wdl_20m.npz...
Constraints configured: 1203 constraints active.
Loaded 19627749 positions.
Label stats: min=0.0000, max=1.0000, mean=0.4998, std=0.3841
  Continuous/mixed labels detected (102 unique values).
Freezing all parameters except: ['PASSED_PATH_SAFE_NONE_ATTACK', 'PASSED_PATH_SAFE_EDGE_ATTACK', 'PASSED_PATH_SAFE_BLOCK_ONLY', 'PASSED_PATH_SUPPORT_BONUS', 'KING_PROXIMITY_MAX_DIST', 'KING_PROX_ENEMY_MULT', 'KING_PROX_ENEMY_DIV', 'KING_PROX_MIN_BONUS', 'KING_PROX_MAX_BONUS']
Active parameters: 9 / 1272
Coordinate auto-scale active: min=1.00, median=1.00, max=2.63
Dev val: 1,000,000  |  train pool: 16,627,749  (dev/holdout never enter mini-batches)
Bucket metrics: 250,000 fixed validation positions (membership from initial theta0)
Initial val MSE (K=1.000): 0.0949271
  Fixed-theta0 validation buckets (MSE, n):
    phase=0               0.0566516  n=74,354

In [9]:
# [Accept] Block 1c
session.accept("block1c_passed_path_prox")

  PASSED_PATH_SAFE_NONE_ATTACK: [11] -> [10]
  PASSED_PATH_SAFE_EDGE_ATTACK: [7] -> [6]
  PASSED_PATH_SAFE_BLOCK_ONLY: [3] -> [2]
  PASSED_PATH_SUPPORT_BONUS: [4] -> [3]
  KING_PROXIMITY_MAX_DIST: [6] -> [7]
  KING_PROX_ENEMY_MULT: [7] -> [8]
  KING_PROX_ENEMY_DIV: [4] -> [3]
  KING_PROX_MIN_BONUS: [-74] -> [-80]
  KING_PROX_MAX_BONUS: [79] -> [81]
Summary: 9 numeric change(s), 100 identical (skipped), 109 names in tuned dump.
Updated 9 constant(s) in chess_engine\classical\constants.py
Next: python -m tuner.sync_constants_snapshot, light tests, then optional tournament vs classical_old.
Wrote tuner\constants_snapshot.py from chess_engine\classical\constants.py; removed 0 stale classical Numba cache file(s)
[block1c_passed_path_prox] Accepted 9 parameter group(s). Backup saved to tuner\runs\backups\20261101_204429_before_block1c_passed_path_prox_constants.py.


['PASSED_PATH_SAFE_NONE_ATTACK',
 'PASSED_PATH_SAFE_EDGE_ATTACK',
 'PASSED_PATH_SAFE_BLOCK_ONLY',
 'PASSED_PATH_SUPPORT_BONUS',
 'KING_PROXIMITY_MAX_DIST',
 'KING_PROX_ENEMY_MULT',
 'KING_PROX_ENEMY_DIV',
 'KING_PROX_MIN_BONUS',
 'KING_PROX_MAX_BONUS']

---
## Block 2：威脅、前哨與棋子協同（對應 `BUG-5` 威脅遮罩、`BUG-6` 弱后、`BUG-7` 前哨修復後）
- **Block 2a**：威脅評估表（`THREAT_*`，對王項已結構性釘死為 0）。
- **Block 2b**：騎士/主教前哨、車開放線、弱后、雙象與輕子協同（`OUTPOST_*`、`ROOK_ON_*`、`WEAK_QUEEN`、`BISHOP_*`、`KING_PROTECTOR`）。

In [ ]:
# [Run] Block 2a — 威脅表 (Threats)
res_2a = session.run_block(
    "block2a_threats",
    tune=[
        "THREAT_SAFE_PAWN",
        "THREAT_KNIGHT_ON_QUEEN",
        "THREAT_SLIDER_ON_QUEEN",
        "THREAT_BY_MINOR",
        "THREAT_BY_ROOK",
        "THREAT_BY_KING",
        "THREAT_HANGING",
        "THREAT_RESTRICTED_PIECE",
        "THREAT_PAWN_PUSH",
    ],
    iters=3500,
    c=2.5,
    alpha=80000.0,
    batch_size=65536,
    report_every=50,
    patience=25,
    seed=201,
)

In [ ]:
# [Accept] Block 2a
session.accept("block2a_threats")

In [ ]:
# [Run] Block 2b — 前哨與棋子協同 (Outposts & Piece Coordination)
res_2b = session.run_block(
    "block2b_outposts_coordination",
    tune=[
        "OUTPOST_BONUS_KNIGHT",
        "OUTPOST_BONUS_BISHOP",
        "REACHABLE_OUTPOST_BONUS",
        "MINOR_BEHIND_PAWN",
        "BISHOP_PAWNS_PENALTY",
        "LONG_DIAGONAL_BISHOP",
        "BISHOP_PAIR_BONUS",
        "BISHOP_PAIR_PAWN_SCALE",
        "ROOK_ON_SEMI_OPEN_FILE_BONUS",
        "ROOK_ON_OPEN_FILE_BONUS",
        "ROOK_ON_QUEEN_FILE",
        "WEAK_QUEEN",
        "KING_PROTECTOR",
    ],
    iters=3500,
    c=2.5,
    alpha=80000.0,
    batch_size=65536,
    report_every=50,
    patience=25,
    seed=202,
)

In [ ]:
# [Accept] Block 2b
session.accept("block2b_outposts_coordination")

---
## Block 3：機動性查表（Mobility Tables — 對應 `BUG-1` 斜向釘死幾何修復後）
每張 Mobility 表的 `0-mobility` (MG/EG) 已錨定於 `theta0`（避免與棋子基礎材質產生常數偏移共線性），並強制單調遞增約束。

In [ ]:
# [Run] Block 3 — 四子機動性查表 (N / B / R / Q Mobility)
res_3 = session.run_block(
    "block3_mobility",
    tune=[
        "KNIGHT_MOBILITY_BONUS",
        "BISHOP_MOBILITY_BONUS",
        "ROOK_MOBILITY_BONUS",
        "QUEEN_MOBILITY_BONUS",
    ],
    iters=4000,
    c=2.5,
    alpha=50000.0,
    batch_size=65536,
    report_every=50,
    patience=30,
    seed=301,
)

In [ ]:
# [Accept] Block 3
session.accept("block3_mobility")

---
## Block 4：兵型結構與連兵（Pawn Structure & Connected Pawns）
- **Block 4a**：孤兵、疊兵、落後兵、弱兵懲罰與連兵等級獎勵表。
- **Block 4b**：連兵支撐乘數（`CONNECTED_SUPPORT_WEIGHT`、`CONNECTED_SUPPORT_WEIGHT_EG`）離散窮舉。

In [ ]:
# [Run] Block 4a — 兵型懲罰與連兵獎勵表
res_4a = session.run_block(
    "block4a_pawn_structure",
    tune=[
        "ISOLATED_PAWN_PENALTY",
        "DOUBLED_PAWN_PENALTY",
        "BACKWARD_PAWN_PENALTY",
        "WEAK_LEVER_PENALTY",
        "WEAK_UNOPPOSED_PENALTY",
        "CONNECTED_BONUS",
        "CONNECTED_BONUS_EG",
    ],
    iters=3500,
    c=2.5,
    alpha=65000.0,
    batch_size=65536,
    report_every=50,
    patience=25,
    seed=401,
)

In [ ]:
# [Accept] Block 4a
session.accept("block4a_pawn_structure")

In [ ]:
# [Run] Block 4b — 連兵支撐權重離散窮舉
res_4b = session.scan_grid(
    "block4b_connected_support_grid",
    grids=[
        "CONNECTED_SUPPORT_WEIGHT=10:12",
        "CONNECTED_SUPPORT_WEIGHT_EG=7:9",
    ],
    split="dev",
    eval_sample=1_000_000,
    top=9,
)

In [ ]:
# [Accept] Block 4b
session.accept("block4b_connected_support_grid")

---
## Block 5：王安全、王前兵盾與王 PST（對應 `BUG-8` King Ring 修復後）
依 `tuner/README.md` 規範，**將 King Danger 與 Shelter + 王 PST 分開調優**，防止三者搶同一組王安全訊號：
- **Block 5a**：`kingDanger` 連續權重（含 `KING_DANGER_NO_QUEEN` 與 `KING_DANGER_NO_QUEEN_ROOKLESS` 雙分流）。
- **Block 5a_grid**：`KING_DANGER_THRESHOLD` 與 `KING_DANGER_OFFSET` 離散閘門掃描。
- **Block 5b**：`SHELTER_*`、`STORM`、`PAWNLESS_FLANK`、`FLANK_ATTACKS`、`TRAPPED_ROOK` 與窄帶王 `PST`（`king_pst_only=True`）。
  > ⚠️ **特別檢查點**：執行 Block 5b 時，請務必確認 `val_pgn.npz` 的 **`castling!=0`** 桶（10.4 萬個真實具易位權局面）沒有顯著惡化！

In [ ]:
# [Run] Block 5a — King Danger 攻擊權重與安全將軍係數
res_5a = session.run_block(
    "block5a_king_danger",
    tune=[
        "KING_ATTACK_WEIGHTS",
        "SAFE_CHECK_QUEEN",
        "SAFE_CHECK_ROOK",
        "SAFE_CHECK_BISHOP",
        "SAFE_CHECK_KNIGHT",
        "KING_DANGER_WEAK_SQ",
        "KING_DANGER_UNSAFE_CHECK",
        "KING_DANGER_BLOCKERS",
        "KING_DANGER_ATTACK_ON_KING_SQ",
        "KING_DANGER_NO_QUEEN",
        "KING_DANGER_NO_QUEEN_ROOKLESS",
        "KING_DANGER_KNIGHT_DEF",
    ],
    iters=3500,
    c=2.5,
    alpha=80000.0,
    batch_size=65536,
    report_every=50,
    patience=25,
    seed=501,
)

In [ ]:
# [Accept] Block 5a
session.accept("block5a_king_danger")

In [ ]:
# [Run] Block 5a_grid — King Danger 啟動門檻與偏置離散掃描
res_5a_grid = session.scan_grid(
    "block5a_king_danger_grid",
    grids=[
        "KING_DANGER_OFFSET=0:4",
        "KING_DANGER_THRESHOLD=66:72",
    ],
    split="dev",
    eval_sample=1_000_000,
    top=10,
)

In [ ]:
# [Accept] Block 5a_grid
session.accept("block5a_king_danger_grid")

In [ ]:
# [Run] Block 5b — 王前兵盾、兵風暴、受困車與窄帶王 PST (請重點檢查 val_pgn castling!=0 桶！)
res_5b = session.run_block(
    "block5b_shelter_storm_king_pst",
    tune=[
        "SHELTER_STRENGTH",
        "UNBLOCKED_STORM",
        "BLOCKED_STORM",
        "BLOCKED_STORM_EG",
        "SHELTER_BASE_MG",
        "SHELTER_BASE_EG",
        "KING_PAWN_DIST_PENALTY_EG",
        "PAWNLESS_FLANK",
        "FLANK_ATTACKS",
        "TRAPPED_ROOK",
        "PST_MG",
        "PST_EG",
    ],
    king_pst_only=True,
    iters=4000,
    c=2.5,
    alpha=55000.0,
    batch_size=65536,
    report_every=50,
    patience=30,
    seed=502,
)

In [ ]:
# [Accept] Block 5b — 確認 val_pgn 的 all 與 castling!=0 桶皆無顯著退步後再套用
session.accept("block5b_shelter_storm_king_pst")

---
## Block 6：單棋種材質 + PST 耦合暖啟動（`N` → `B` → `R` → `Q` → `P`）
- `PST_MG` / `PST_EG` 每列固定 `theta0` 平均值（量化整數和守恆），因此格間形狀由 PST 負責、整體棋種基準值由 Material 負責。
- `MIDGAME_LIMIT` / `ENDGAME_LIMIT` 隨候選 MG `N/B/R/Q` 動態連動，確保開局永遠對齊 `phase = 128`。
- `Block 6e (Pawn)` 僅開 `EG_MATERIAL_VALUES[PAWN]`（±10% 軟帶）與兵 `PST`（Rank 2–7），`MG_MATERIAL_VALUES[PAWN] = 100` 永遠鎖定作為全引擎分值錨點。

In [ ]:
# [Run] Block 6a_knight — 騎士 (Knight) PST 形狀調優 (子力交換比率保持平衡)
res_6a_knight = session.run_block(
    "block6a_knight",
    tune=["PST_MG", "PST_EG"],
    pst_pieces="N",
    iters=3500,
    c=2.5,
    alpha=50000.0,
    batch_size=65536,
    report_every=50,
    patience=25,
    seed=601,
)


In [ ]:
# [Accept] Block 6a_knight
session.accept("block6a_knight")

In [ ]:
# [Run] Block 6b_bishop — 主教 (Bishop) PST 形狀調優 (子力交換比率保持平衡)
res_6b_bishop = session.run_block(
    "block6b_bishop",
    tune=["PST_MG", "PST_EG"],
    pst_pieces="B",
    iters=3500,
    c=2.5,
    alpha=50000.0,
    batch_size=65536,
    report_every=50,
    patience=25,
    seed=602,
)


In [ ]:
# [Accept] Block 6b_bishop
session.accept("block6b_bishop")

In [ ]:
# [Run] Block 6c_rook — 城堡 (Rook) PST 形狀調優 (子力交換比率保持平衡)
res_6c_rook = session.run_block(
    "block6c_rook",
    tune=["PST_MG", "PST_EG"],
    pst_pieces="R",
    iters=3500,
    c=2.5,
    alpha=50000.0,
    batch_size=65536,
    report_every=50,
    patience=25,
    seed=603,
)


In [ ]:
# [Accept] Block 6c_rook
session.accept("block6c_rook")

In [ ]:
# [Run] Block 6d_queen — 皇后 (Queen) PST 形狀調優 (子力交換比率保持平衡)
res_6d_queen = session.run_block(
    "block6d_queen",
    tune=["PST_MG", "PST_EG"],
    pst_pieces="Q",
    iters=3500,
    c=2.5,
    alpha=50000.0,
    batch_size=65536,
    report_every=50,
    patience=25,
    seed=604,
)


In [ ]:
# [Accept] Block 6d_queen
session.accept("block6d_queen")

In [ ]:
# [Run] Block 6e_pawn — 士兵 PST 形狀調優 (MG Pawn 固定 100cp 錨點)
res_6e_pawn = session.run_block(
    "block6e_pawn",
    tune=["PST_MG", "PST_EG"],
    pst_pieces="P",
    iters=3500,
    c=2.5,
    alpha=50000.0,
    batch_size=65536,
    report_every=50,
    patience=25,
    seed=605,
)


In [ ]:
# [Accept] Block 6e_pawn
session.accept("block6e_pawn")

---
## Block 7：材質不平衡、主動權與空間/先手微調（Imbalance, Initiative, Space & Tempo）
- **Block 7a**：二次材質不平衡矩陣（結構性 0 已自動釘死並排除出 SPSA mask，僅調非零交互項，±10% 軟帶）。
- **Block 7b**：Initiative 複雜度權重與偏置（未使用的 `INITIATIVE_PAWN_ENDGAME_WEIGHT_*` 已自動凍結）。
- **Block 7c_grid**：`TEMPO_BONUS` 與少數離散除數（`SPACE_*_DIVISOR`、`IMBALANCE_DIVISOR`）窮舉掃描。

In [ ]:
# [Run] Block 7a — 二次材質不平衡矩陣 (Imbalance Polynomial)
res_7a = session.run_block(
    "block7a_imbalance",
    tune=[
        "IMBALANCE_QUADRATIC_OURS",
        "IMBALANCE_QUADRATIC_THEIRS",
        "IMBALANCE_SCALE_MG",
        "IMBALANCE_SCALE_EG",
    ],
    iters=3000,
    c=2.0,
    alpha=60000.0,
    batch_size=65536,
    report_every=50,
    patience=25,
    seed=701,
)

In [ ]:
# [Accept] Block 7a
session.accept("block7a_imbalance")

In [ ]:
# [Run] Block 7b — 主動權複雜度係數 (Initiative / Complexity)
res_7b = session.run_block(
    "block7b_initiative",
    tune=[
        "INITIATIVE_PASSED_WEIGHT_MG",
        "INITIATIVE_PAWN_WEIGHT_MG",
        "INITIATIVE_OUTFLANKING_WEIGHT_MG",
        "INITIATIVE_INFILTRATION_WEIGHT_MG",
        "INITIATIVE_BOTH_FLANKS_WEIGHT_MG",
        "INITIATIVE_ALMOST_UNWIN_WEIGHT_MG",
        "INITIATIVE_OFFSET_MG",
        "INITIATIVE_MG_OFFSET",
        "INITIATIVE_PASSED_WEIGHT_EG",
        "INITIATIVE_PAWN_WEIGHT_EG",
        "INITIATIVE_OUTFLANKING_WEIGHT_EG",
        "INITIATIVE_INFILTRATION_WEIGHT_EG",
        "INITIATIVE_BOTH_FLANKS_WEIGHT_EG",
        "INITIATIVE_ALMOST_UNWIN_WEIGHT_EG",
        "INITIATIVE_OFFSET_EG",
    ],
    iters=2500,
    c=2.0,
    alpha=35000.0,
    batch_size=65536,
    report_every=50,
    patience=20,
    seed=702,
)

In [ ]:
# [Accept] Block 7b
session.accept("block7b_initiative")

In [ ]:
# [Run] Block 7c_grid — Tempo、Space 門檻與除數離散掃描
# 註：TEMPO_BONUS 與搜尋深度耦合較強，若靜態改善極微弱建議維持原值，留待對局驗證決定。
res_7c = session.scan_grid(
    "block7c_space_tempo_grid",
    grids=[
        "SPACE_BONUS_DIVISOR=16:18",
        "SPACE_SCALE_DIVISOR=8:10",
        "IMBALANCE_DIVISOR=20:22",
        "SPACE_THRESHOLD=5300:5500:100",
        "TEMPO_BONUS=15:17",
    ],
    split="dev",
    eval_sample=1_000_000,
    top=10,
)

In [ ]:
# [Accept] Block 7c_grid — 可用 only=[...] 只挑確認有效的常數套用
session.accept("block7c_space_tempo_grid")

---
## Block 8：全子力聯合收尾 Polish（`NBRQ` Material + PST Joint Polish）
在前面各獨立區塊已收斂並接受後，用較小學習率 `alpha=20000, c=2.0` 同時打開 `N/B/R/Q` 的材質與 PST，消除跨區塊之間的殘餘交互作用。

In [ ]:
# [Run] Block 8 — NBRQ Material + PST 聯合 Polish
res_8 = session.run_block(
    "block8_nbrq_joint_polish",
    tune=["MG_MATERIAL_VALUES", "EG_MATERIAL_VALUES", "PST_MG", "PST_EG"],
    pst_pieces="NBRQ",
    iters=3500,
    c=2.0,
    alpha=20000.0,
    batch_size=65536,
    report_every=50,
    patience=30,
    seed=801,
)

In [ ]:
# [Accept] Block 8
session.accept("block8_nbrq_joint_polish")

---
## 9. 總結驗收、回滾工具與對戰測試指令
完成一輪（或數個區塊）的調參後：
1. 執行下方 **`session.cumulative_report()`** 檢視相較於本次啟動前初始 `theta0` 的累積變動與兩個獨立驗證集總增益。
2. 執行輕量 **Parity / Conformance 單元測試**，確保正式引擎與 tunable 評估完全一致且殘局不變量通過。
3. **正式宣稱 Elo 提升前**，請在終端機執行 `tools/tournament.py` 與 `classical_old` 對戰驗證（注意：套用新參數**後**切勿執行 `sync_classical_old.py --sync`，直到對戰驗收通過！）。

In [ ]:
# 9a. 累積總成績單（目前 constants.py 對比本次 Session 啟動前的初始 theta0）
cum_stats = session.cumulative_report()

In [ ]:
# 9b. 【選用工具】回滾上一步 或 手動覆寫單一參數重新驗證
# 若要撤銷最近一次 accept()：
# session.rollback()

# 若要撤銷到特定區塊之前的備份（例如撤銷 block5b 及其之後的 accept）：
# session.rollback("block5b_shelter_storm_king_pst")

# 若要測試某個候選檔加上手動微調單一值：
# session.validate_file("block5a_king_danger", overrides=["KING_DANGER_NO_QUEEN=570"])

In [ ]:
# 9c. 同步 tunable_eval 並執行評估一致性與殘局不變量單元測試
import subprocess

cmds = [
    [sys.executable, "-m", "tuner.sync_constants_snapshot"],
    [sys.executable, "-m", "tuner.codegen.generate_tunable_eval"],
    [
        sys.executable, "-m", "unittest",
        "tests.test_texel_parameterization",
        "tests.test_endgame_conformance",
        "tests.test_tunable_eval_parity",
        "tests.test_runtime_eval_weights",
        "-v",
    ],
]
for cmd in cmds:
    print("+", " ".join(str(x) for x in cmd))
    rc = subprocess.call(cmd, cwd=str(REPO_ROOT))
    if rc != 0:
        raise RuntimeError(f"Command failed with exit code {rc}: {cmd}")
print("\n✅ 所有一致性與殘局約束測試皆通過！可進行對局閘門驗證：")
print("   python tools/tournament.py -c 4 --games 500 --nodes 300000 --name1 New --name2 Old")